In [0]:
from pyspark.sql.functions import (
    avg,
    col,
    current_timestamp,
    lag,
    max as spark_max,
    round as spark_round,
    sum as spark_sum,
    when
)
from pyspark.sql.window import Window
from delta.tables import DeltaTable

CATALOG_NAME = "isp"
SILVER_SCHEMA_NAME = "silver"
GOLD_SCHEMA_NAME = "gold"
VOLUME_NAME = "isp_volumes"
SOURCE_TABLE = "timeseries_monthly_since_2003"

silver_table = f"{CATALOG_NAME}.{SILVER_SCHEMA_NAME}.{SOURCE_TABLE}"
gold_monthly_table = f"{CATALOG_NAME}.{GOLD_SCHEMA_NAME}.fact_monthly_crime_rates"
gold_annual_table = f"{CATALOG_NAME}.{GOLD_SCHEMA_NAME}.agg_annual_crime_summary"
checkpoint_path = f"/Volumes/{CATALOG_NAME}/{GOLD_SCHEMA_NAME}/{VOLUME_NAME}/_checkpoints/fact_monthly_crime_rates"

In [0]:
def map_crime_category(df):
    """Categorizes granular occurrences into high-level analytical dimensions."""
    return df.withColumn(
        "categoria_ocorrencia",
        when(
            col("ocorrencia").isin(
                "hom_doloso", "lesao_corp_morte", "latrocinio", 
                "cvli", "letalidade_violenta", "tentat_hom", "hom_por_interv_policial"
            ),
            "Crimes Contra a Vida e CVLI"
        )
        .when(
            col("ocorrencia").like("%roubo%"),
            "Crimes Contra o Patrimônio (Roubos)"
        )
        .when(
            col("ocorrencia").like("%furto%"),
            "Crimes Contra o Patrimônio (Furtos)"
        )
        .when(
            col("ocorrencia").isin("hom_culposo", "lesao_corp_culposa"),
            "Crimes Culposos / Acidentes"
        )
        .when(
            col("ocorrencia").isin("apreensao_drogas", "posse_drogas", "trafico_drogas", "apreensao_drogas_sem_autor"),
            "Entorpecentes e Drogas"
        )
        .otherwise("Outras Ocorrências e Registros")
    )

In [0]:
df_silver_stream = (
    spark.readStream
    .format("delta")
    .table(silver_table)
)

def upsert_gold_fact(micro_batch_df, batch_id):
    if micro_batch_df.isEmpty():
        return

    df_categorized = map_crime_category(micro_batch_df)

    window_rolling_3m = (
        Window.partitionBy("ocorrencia")
        .orderBy("reference_date")
        .rowsBetween(-2, 0)
    )
    window_lag_12m = (
        Window.partitionBy("ocorrencia")
        .orderBy("reference_date")
    )

    df_gold_fact = (
        df_categorized
        .withColumn(
            "media_movel_3m",
            spark_round(avg("taxa").over(window_rolling_3m), 2)
        )
        .withColumn(
            "taxa_ano_anterior",
            lag("taxa", 12).over(window_lag_12m)
        )
        .withColumn(
            "variacao_yoy_perc",
            when(
                col("taxa_ano_anterior").isNotNull() & (col("taxa_ano_anterior") > 0),
                spark_round(((col("taxa") - col("taxa_ano_anterior")) / col("taxa_ano_anterior")) * 100, 2)
            ).otherwise(None)
        )
        .withColumn("_gold_processed_at", current_timestamp())
    )

    if not spark.catalog.tableExists(gold_monthly_table):
        (
            df_gold_fact.limit(0)
            .write
            .format("delta")
            .saveAsTable(gold_monthly_table)
        )

    gold_delta = DeltaTable.forName(spark, gold_monthly_table)
    (
        gold_delta.alias("target")
        .merge(
            df_gold_fact.alias("source"),
            """
            target.ano = source.ano 
            AND target.mes = source.mes 
            AND target.ocorrencia = source.ocorrencia
            """
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

query_gold_fact = (
    df_silver_stream
    .writeStream
    .format("delta")
    .foreachBatch(upsert_gold_fact)
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .start()
)

query_gold_fact.awaitTermination()

In [0]:
df_gold_source = spark.table(gold_monthly_table)

df_annual_summary = (
    df_gold_source
    .groupBy("ano", "categoria_ocorrencia", "ocorrencia")
    .agg(
        spark_round(avg("taxa"), 2).alias("taxa_media_mensal"),
        spark_round(spark_max("taxa"), 2).alias("taxa_maxima_mensal"),
        spark_round(spark_sum("taxa"), 2).alias("taxa_acumulada_anual")
    )
    .withColumn("_gold_processed_at", current_timestamp())
)

(
    df_annual_summary
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold_annual_table)
)

In [0]:
%sql

SELECT * 
FROM isp.gold.fact_monthly_crime_rates 
WHERE ocorrencia = 'letalidade_violenta'
ORDER BY reference_date DESC;

In [0]:
%sql
SELECT * 
FROM isp.gold.agg_annual_crime_summary 
WHERE ocorrencia = 'roubo_veiculo'
ORDER BY ano DESC;